# Jupyter Notebook: SLC9000 API #

### Notes:

#### 2026-06-06: SLC9000 firmware 9.7.0.0R11 most of the API functionality not working
#### 2026-06-08: SLC9000 firmware 9.7.0.0R16 API working - still get a timeout on PUT operations (cURL commands work)

## Import modules and define connection settings

This cell sets up the environment for talking to the SLC9000 device. It imports required Python libraries, reads credentials from environment variables, configures the `requests` session (TLS verification, headers, timeout), and defines the base URL and other constants used by the REST API calls.


In [25]:
import json
import os
import requests
import time
from pprint import pprint

TIMEOUT = 10
VERIFY = False

# Create & use these environment variables or the load_config function
USERNAME = os.environ["SLC9K_USER"]
PASSWORD = os.environ["SLC9K_PW"]
#
requests.packages.urllib3.disable_warnings()
session = requests.Session()
session.verify = VERIFY
session.timeout = TIMEOUT
session.headers.update(
    {
        "Content-Type": "application/json",
        "Accept": "application/json",
   }
)
session.base_url = "https://10.40.21.41"

### Load configuration from JSON file

In [26]:
def load_config(filename="config.json"):
    """
    Load connection and authentication settings from a JSON config file.

    This function reads the given JSON configuration file and updates the
    global TIMEOUT, VERIFY, USERNAME, and PASSWORD variables. It allows the
    SLC9000 API demo to be configured without hard‑coding values in the
    notebook or relying solely on environment variables.

    The expected JSON structure is:
        {
            "TIMEOUT": 10,
            "VERIFY": false,
            "USERNAME": "sysadmin",
            "PASSWORD": "ciscolive"
        }

    Args:
        filename: Path to the JSON configuration file to load. Defaults to
            "config.json" in the current working directory.
    """
    global TIMEOUT, VERIFY, USERNAME, PASSWORD
    with open(filename, "r") as f:
        config = json.load(f)

    TIMEOUT = config.get("TIMEOUT", TIMEOUT)
    VERIFY = config.get("VERIFY", VERIFY)
    USERNAME = config.get("USERNAME", USERNAME)
    PASSWORD = config.get("PASSWORD", PASSWORD)

## Helper functions for SLC9000 REST API calls

In [27]:
def api_call(
    method: str,
    path: str,
    payload: dict | None = None
):
    """
    Perform an HTTP request to the SLC9000 API using the shared session.

    This helper builds a full URL from the session's base_url and the given
    path, sends the request with the specified HTTP method, and handles basic
    error reporting. On success it returns the underlying requests.Response
    object; on failure it logs the error and returns None.

    Args:
        method: HTTP method to use, e.g. "GET", "POST", "PUT", or "DELETE".
        path: API path to append to session.base_url, e.g. "/api/v2/system/status".
        payload: Optional JSON-serializable dictionary to send as the request body
            for POST, PUT, and DELETE requests.

    Returns:
        A requests.Response object if the request succeeds; otherwise None.

    Raises:
        ValueError: If an unsupported HTTP method is provided.
    """
    url = f"{session.base_url}{path}"
    try:
        if method == "GET":
            response = session.get(url, timeout=TIMEOUT)
        elif method == "POST":
            response = session.post(url, json=payload or {}, timeout=TIMEOUT)
        elif method == "PUT":
            response = session.put(url, json=payload or {}, timeout=TIMEOUT)
        elif method == "DELETE":
            response = session.delete(url, json=payload or {}, timeout=TIMEOUT)
        else:
            raise ValueError(f"Unsupported method: {method}")

        response.raise_for_status()
        print(f"{url} successful ({response.status_code})")
        return response
    except requests.exceptions.HTTPError as http_err:
        body = response.text if "response" in locals() else "<no response>"
        print(f"HTTP error: {http_err} - Response: {body}")
    except requests.exceptions.RequestException as err:
        print(f"Request error: {err}")
    return None


# User Management
def user_login(username: str, password: str):
    """
    User login for all roles
    """
    response = api_call(
        "POST",
        "/api/v2/user/login",
        {"username": username, "password": password},
    )

    if response is None:
        return None

    data = response.json()
    token = data.get("token")
    if token:
        session.headers.update({"X-auth-token": token})
    print(token)
    return response

def sessions():
    """
    Get active sessions
    """
    return api_call("GET", "/api/v2/sessions")

def sessions_delete(session_id: int | str):
    """
    Terminate a session
    """
    return api_call("DELETE", f"/api/v2/sessions/{session_id}")

def user_logout():
    """
    Logout from an API session
    """
    return api_call("DELETE", "/api/v2/user/login")


# System
def system_identity():
    """
    Get basic system info
    """
    return api_call("GET", "/api/v2/system/identity")

def system_reboot():
    """
    Reboot system
    """
    return api_call("POST", "/api/v2/system/reboot")

def system_status():
    """
    Get system status
    """
    return api_call("GET", "/api/v2/system/status")

def system_version():
    """
    Get HW/SW versions
    """
    return api_call("GET", "/api/v2/system/version")

def system_ztp():
    """
    Get ZTP/bootstrap Status
    """
    return api_call("GET", "/api/v2/system/ztp")


# Network
def network_interfaces():
    """
    Get basic network info
    """
    return api_call("GET", "/api/v2/network/interfaces")

def network_interfaces_set(payload: dict | None = None):
    """
    Get basic network info
    """
    return api_call("PUT", "/api/v2/network/interfaces", data=payload)


# Firmware
def firmware_version():
    """
    Get Firmware Version
    """
    return api_call("GET", "/api/v2/firmware/version")

def firmware_bootbank():
    """
    Get current boot bank
    """
    return api_call("GET", "/api/v2/firmware/bootbank")

def firmware_check():
    """
    Check for firmware updates
    """
    return api_call("GET", "/api/v2/firmware/check")

def firmware_bootbank_set(payload: dict | None = None):
    """
    Set active boot bank
    Example payload:
      {"bank": 1}    
    """
    return api_call("PUT", "/api/v2/firmware/bootbank", payload=payload)

def firmware_log():
    """
    Get FW Update Log
    """
    return api_call("GET", "/api/v2/firmware/log")


# Config
def config_commands():
    """
    Get config (CLI commands)
    """
    return api_call("GET", "/api/v2/config/commands")


# Ports
def ports():
    """
    Get all serial ports status
    """
    return api_call("GET", "/api/v2/ports")

def port_status(port_id: str = "1"):
    """
    Get serial port status
    """
    return api_call("GET", f"/api/v2/ports/{port_id}/status")

def connections():
    """
    Get serial connections
    """
    return api_call("GET", "/api/v2/connections")


# Cellular
def cellular_status():
    return api_call("GET", "/api/v2/cellular/status")
    

# Managed
def managed_devices():
    # return api_call("GET", "/api/v2/managed_devices?connection_filter=serial")
    return api_call("GET", "/api/v2/managed_devices")


### Initialize global variables

In [28]:
load_config()

print(f"TIMEOUT={TIMEOUT}")
print(f"VERIFY={VERIFY}")

TIMEOUT=10
VERIFY=False


## Authenticate and obtain API session token

This section logs in to the SLC9000 using the provided username and password. On success, the device returns a JSON response containing a session `token`, its `expires_in` lifetime (in seconds), and user details. The token is used by the REST API to identify and authorize subsequent requests made during this session.

In [29]:
login = user_login(username=USERNAME, password=PASSWORD)

if login is not None:
    pprint(login.json())

https://10.40.21.41/api/v2/user/login successful (200)
e4c8f5e8-5aaf-4c06-bf75-de8476b46cc3
{'authenticated': 'Local Users',
 'expires_in': 1800,
 'token': 'e4c8f5e8-5aaf-4c06-bf75-de8476b46cc3',
 'user': {'allow_dialback': False,
          'break_seq': '\\x1bB',
          'clear_ports': '1-32,U1,U2',
          'data_ports': '1-32,U1,U2',
          'dialback_number': 'null',
          'escape_seq': '\\x1bA',
          'group': 'Administrators',
          'listen_ports': '1-32,U1,U2',
          'permissions': 'ad,nt,sv,dt,lu,ra,um,dp,ub,rs,fc,dr,sn,wb,sk,po,do,md,rp,sw',
          'power_outlets': '1-8',
          'uid': 0,
          'username': 'sysadmin'}}


### List active API sessions

In [30]:
get_active_sessions = sessions()
sessions_data = get_active_sessions.json()
results = sessions_data["results"]
sorted_results = sorted(results, key=lambda s: s["login_time"], reverse=False)
pprint(sorted_results)

https://10.40.21.41/api/v2/sessions successful (200)
[{'id': 7490,
  'idle_time': '00:00:00:00',
  'login_time': '2026-06-19T06:28:04Z',
  'remote_ip': '10.40.21.223',
  'session_type': 'REST API',
  'username': 'sysadmin'}]


### Check current firmware boot bank

In [42]:
get_firmware_bootbank = firmware_bootbank()

if get_firmware_bootbank is not None:
    pprint(get_firmware_bootbank.json())

https://10.40.21.41/api/v2/firmware/bootbank successful (200)
{'bank': 1}


### Inspect firmware update log

In [ ]:
get_firmware_log = firmware_log()

if get_firmware_log is not None:
    pprint(get_firmware_log.json())

### Attempt to query firmware version endpoint

In [ ]:
get_firmware_check = firmware_check()

if get_firmware_check is not None:
    pprint(get_firmware_check.json())

### Retrieve system software versions

In [ ]:
get_system_version = system_version()

if get_system_version is not None:
    pprint(get_system_version.json())

### Check Zero Touch Provisioning (ZTP) status

In [ ]:
get_system_ztp = system_ztp()

if get_system_ztp is not None:
    pprint(get_system_ztp.json())

### Monitor system health and hardware status

In [ ]:
get_system_status = system_status()

if get_system_status is not None:
    pprint(get_system_status.json())

### Show device identity and metadata

In [ ]:
get_system_identity = system_identity()

if get_system_identity is not None:
    pprint(get_system_identity.json())

### Display network interface configuration

In [ ]:
get_network_interfaces = network_interfaces()

if get_network_interfaces is not None:
    pprint(get_network_interfaces.json())

### Log out and terminate API session

In [ ]:
logout = user_logout()

if logout is not None:
    pprint(logout.json())

### Others...

In [ ]:
get_config_commands = config_commands()

if get_config_commands is not None:
    pprint(get_config_commands.json())

In [ ]:
post_system_reboot = system_reboot()

if post_system_reboot is not None:
    pprint(post_system_reboot.json())

In [36]:
get_cellular = cellular_status()

if get_cellular is not None:
    pprint(get_cellular.json())

https://10.40.21.41/api/v2/cellular/status successful (200)
{'apn': 'null',
 'band': '',
 'carrier': 'null',
 'connection': 'null',
 'country_operator': 'null',
 'current_band': 'null',
 'dns_server1': 'null',
 'dns_server2': 'null',
 'firmware_revision': 'null',
 'gateway': 'null',
 'hardware_revision': 'null',
 'iccid': 'null',
 'imei': 'null',
 'ipv4_address': 'null',
 'ipv4_mask': 'null',
 'ipv6_global': 'null',
 'ipv6_state': True,
 'ipv6_static': 'null',
 'link_state': 'down',
 'model': 'null',
 'modem': False,
 'network_registration': 'null',
 'os_version': 'null',
 'packet_data_state': 'down',
 'pdp_context_id': 1,
 'preferred_network': 'auto',
 'roaming_state': False,
 'roaming_status': False,
 'rx_bytes': 0,
 'rx_errors': 0,
 'rx_multicast': 0,
 'rx_packets': 0,
 'serial_number': 'null',
 'signal_strength': 0,
 'sim_card': 'not_inserted',
 'state': 'DHCP',
 'tx_bytes': 0,
 'tx_errors': 0,
 'tx_packets': 0,
 'uptime': 0}


In [37]:
get_ports = ports()

if get_ports is not None:
    print(f"Total Ports: {get_ports.json()['total_ports']}")
    print("First 3...")
    pprint(get_ports.json()['ports'][0:2])
    print(f"{int(get_ports.json()['total_ports'])-3} not displayed...")

https://10.40.21.41/api/v2/ports successful (200)
Total Ports: 32
First 3...
[{'bytes_input': 0,
  'bytes_output': 0,
  'cts': True,
  'dsr': True,
  'dtr': True,
  'errors': 0,
  'id': 1,
  'md_name': '',
  'md_type': '',
  'name': 'MikroTik',
  'rts': True,
  'status': 'Idle',
  'type': 'rj45'},
 {'bytes_input': 21377733,
  'bytes_output': 207089,
  'cts': True,
  'dsr': True,
  'dtr': True,
  'errors': 0,
  'id': 2,
  'md_name': '',
  'md_type': '',
  'name': 'C892FSP-K9',
  'rts': True,
  'status': 'Idle',
  'type': 'rj45'}]
29 not displayed...


In [38]:
get_port_status = port_status(port_id=2)

if get_port_status is not None:
    pprint(get_port_status.json())

https://10.40.21.41/api/v2/ports/2/status successful (200)
{'bytes_input': 21377733,
 'bytes_output': 207089,
 'cts': True,
 'dsr': True,
 'dtr': True,
 'errors': 0,
 'id': 2,
 'md_name': '',
 'md_type': '',
 'name': 'C892FSP-K9',
 'rts': True,
 'status': 'Idle',
 'type': 'rj45'}


In [39]:
get_connections = connections()

if get_connections is not None:
    total_connected = len(get_connections.json()['list'])
    print(f"Total connections: {total_connected}...")
    pprint(get_connections.json()['list'])

https://10.40.21.41/api/v2/connections successful (200)
Total connections: 1...
[{'description': 'Console Port to Command Line',
  'direction': 'bi-directional',
  'duration': 131455,
  'id': 2,
  'idle_time': 0,
  'source_ip': '',
  'status': 'connected (waiting)',
  'username': ''}]


In [40]:
get_managed_devices = managed_devices()

if get_managed_devices is not None:
    pprint(get_managed_devices.json())

https://10.40.21.41/api/v2/managed_devices successful (200)
{'devices': [{'connection_type': 'serial',
              'cpu_usage': 0,
              'device_banner': 'null',
              'hostname': 'router',
              'last_operation': 'null',
              'last_operation_date': 'null',
              'last_seen_date': '2026-06-19T06:29:10',
              'management_ipv4': '10.40.21.205',
              'memory_usage': 0,
              'model': 'C892FSP-K9',
              'name': 'router',
              'onboard_date': '2026-06-10T12:41:56',
              'os_version': '15.7(3)M',
              'port_id': 2,
              'serial_number': 'FJC2402L07T',
              'temperature': 0,
              'type': 'cisco ios',
              'uptime': 155820}],
 'total_devices': 1}


In [41]:
firmware_bootbank_set(payload={"bank": 1})

Request error: HTTPSConnectionPool(host='10.40.21.41', port=443): Read timed out. (read timeout=10)


# Working around PUT issue #

## This times out ##

In [ ]:
token = session.headers["X-Auth-Token"]
url = "https://10.40.21.41/api/v2/firmware/bootbank"
body = '{"bank": 1}'

headers = {
    "Host": "10.40.21.41",
    "User-Agent": "curl/8.7.1",
    "Content-Type": "application/json",
    "Accept": "application/json",
    "X-Auth-Token": token,
    "Accept-Encoding": "identity",  # avoid gzip/deflate differences
    "Connection": "close",
    "Content-Length": str(len(body)),
}

response = requests.put(
    url,
    headers=headers,
    data=body,
    timeout=60,
    verify=False,
)

print("Status:", response.status_code)
print("Body:", response.text)
print("Sent headers:\n", response.request.headers)
print("Sent body:\n", response.request.body)

In [ ]:
token = session.headers["X-Auth-Token"]

url = "https://10.40.21.41/api/v2/firmware/bootbank"

payload = { "bank": 1 }
headers = {
    "Content-Type": "application/json",
    "Accept": "application/json",
    "X-auth-token": token
}

response = requests.put(url, json=payload, headers=headers, verify=False,)

print(response.json())

## This works ##

In [ ]:
import subprocess
import json

def firmware_bootbank_set_via_curl(bank: int, token: str):
    cmd = [
        "curl",
        "-k",
        "-X", "PUT",
        "-H", "Content-Type: application/json",
        "-H", "Accept: application/json",
        f"-H", f"X-Auth-Token:{token}",
        "-d", json.dumps({"bank": bank}),
        "https://10.40.21.41/api/v2/firmware/bootbank",
    ]
    result = subprocess.run(cmd, capture_output=True, text=True, check=False)
    return result.stdout, result.stderr, result.returncode

In [ ]:
change_bootbank = firmware_bootbank_set_via_curl(2, session.headers['X-Auth-Token'])
for _ in change_bootbank:
    print(_)